# 04-6. JSON·JSON Lines와 직렬화 검증 실습

## Goal

Python 값과 JSON 값의 대응을 확인하고, 문법·중복 키·비표준 숫자·업무 스키마를 구분해 검증합니다. JSON 문서와 JSON Lines의 처리 경계를 비교하며 각 단계는 **결과 예측 → 실행 → 이유 설명 → 입력 변경** 순서로 진행하세요.


## Setup

모든 JSON과 JSON Lines 파일은 임시 디렉터리에 만듭니다. 신뢰하지 않는 pickle은 읽지 않으며 Python 3.10 이상의 표준 라이브러리만 사용합니다.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import math

json_tempdir = TemporaryDirectory(prefix="python-04-6-")
lab_dir = Path(json_tempdir.name)
assert lab_dir.is_dir()


def expect_exception(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type as exc:
        return exc
    raise AssertionError(f"{exception_type.__name__}이 발생해야 합니다")


print("격리 실습 디렉터리:", lab_dir)


## Steps

### 1. Python과 JSON 자료형의 왕복

tuple은 JSON array로 기록되고 다시 읽으면 list가 됩니다. 직렬화 왕복이 원래 Python 자료형을 모두 보존한다고 가정하지 않습니다.


In [ ]:
source = {
    "position": (10, 20),
    "enabled": True,
    "note": None,
}
serialized_source = json.dumps(source)
restored_source = json.loads(serialized_source)

assert restored_source == {
    "position": [10, 20],
    "enabled": True,
    "note": None,
}
assert isinstance(restored_source["position"], list)
assert type(restored_source["enabled"]) is bool
assert restored_source["note"] is None

print(restored_source)


### 2. dumps·loads와 dump·load 구분

이름 끝의 s가 있는 함수는 문자열을, 없는 함수는 열린 텍스트 파일 객체를 대상으로 합니다. UTF-8과 표준 JSON 숫자 정책을 명시합니다.


In [ ]:
record = {
    "name": "Alice",
    "active": True,
    "score": 91,
    "tags": ["python", "file"],
}

record_text = json.dumps(
    record,
    ensure_ascii=False,
    indent=2,
    allow_nan=False,
)
restored_record = json.loads(record_text)

record_path = lab_dir / "record.json"
with record_path.open("w", encoding="utf-8") as file:
    json.dump(
        record,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )
    file.write("\n")

with record_path.open("r", encoding="utf-8") as file:
    loaded_record = json.load(file)

assert restored_record == record
assert loaded_record == record
assert record_path.read_bytes().endswith(b"\n")
assert "Alice" in record_path.read_text(encoding="utf-8")

print(record_text)


### 3. JSON 문법 오류의 위치

파싱 오류에서는 행·열·문자 위치와 일반화한 원인을 확인합니다. 토큰이나 개인정보가 있을 수 있는 입력 전문은 오류 기록에 복사하지 않습니다.


In [ ]:
invalid_json_text = """{
  "name": "Alice",
  "active": true,
}"""

syntax_error = expect_exception(
    json.JSONDecodeError,
    json.loads,
    invalid_json_text,
)

assert syntax_error.lineno >= 1
assert syntax_error.colno >= 1
assert isinstance(syntax_error.pos, int)
assert syntax_error.msg
syntax_error_summary = {
    "line": syntax_error.lineno,
    "column": syntax_error.colno,
    "error": syntax_error.msg,
}
assert invalid_json_text not in str(syntax_error_summary)

print(syntax_error_summary)


### 4. 중복 키와 비표준 숫자 거부

기본 디코더가 허용할 수 있는 중복 객체 키와 NaN·Infinity를 입력 계약에서 명시적으로 거부합니다.


In [ ]:
def reject_nonstandard_constant(value):
    raise ValueError(f"표준 JSON 숫자가 아닙니다: {value}")


def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키입니다: {key}")
        result[key] = value
    return result


def loads_strict(text):
    return json.loads(
        text,
        parse_constant=reject_nonstandard_constant,
        object_pairs_hook=reject_duplicate_keys,
    )


def load_strict(file):
    return json.load(
        file,
        parse_constant=reject_nonstandard_constant,
        object_pairs_hook=reject_duplicate_keys,
    )


duplicate_error = expect_exception(
    ValueError,
    loads_strict,
    '{"role": "user", "role": "admin"}',
)
nan_input_error = expect_exception(
    ValueError,
    loads_strict,
    '{"score": NaN}',
)
infinity_input_error = expect_exception(
    ValueError,
    loads_strict,
    '{"score": Infinity}',
)

assert "중복 JSON 키" in str(duplicate_error)
assert "NaN" in str(nan_input_error)
assert "Infinity" in str(infinity_input_error)

print("엄격 입력 정책 3종을 확인했습니다.")


### 5. 지원하지 않는 값과 명시적 변환

datetime, Path, set은 기본 JSON 자료형이 아닙니다. default=str로 오류를 숨기지 말고 교환 계약에 맞는 문자열이나 배열로 명시적으로 바꿉니다.


In [ ]:
unsupported = {
    "created_at": datetime(2026, 9, 1, tzinfo=timezone.utc),
    "path": Path("reports/result.json"),
    "labels": {"reviewed", "safe"},
}
unsupported_error = expect_exception(
    TypeError,
    json.dumps,
    unsupported,
)

converted = {
    "created_at": unsupported["created_at"].isoformat(),
    "path": unsupported["path"].as_posix(),
    "labels": sorted(unsupported["labels"]),
}
converted_text = json.dumps(
    converted,
    ensure_ascii=False,
    allow_nan=False,
)
assert loads_strict(converted_text) == converted
assert "serializable" in str(unsupported_error)

colliding_key_text = json.dumps({1: "integer", "1": "string"})
colliding_key_error = expect_exception(
    ValueError,
    loads_strict,
    colliding_key_text,
)
assert "중복 JSON 키" in str(colliding_key_error)

print(converted)


### 6. 파싱 뒤 업무 스키마 검증

JSON 파싱 성공과 업무 데이터 유효성은 별개입니다. 필수 키·알 수 없는 키·자료형·빈 값·숫자 범위를 확인하고 정규화한 새 딕셔너리를 반환합니다.


In [ ]:
REQUIRED_KEYS = {"name", "active", "score", "tags"}


def validate_record(data):
    if not isinstance(data, dict):
        raise TypeError("최상위 값은 object여야 합니다")

    keys = set(data)
    missing = REQUIRED_KEYS - keys
    unknown = keys - REQUIRED_KEYS

    if missing:
        raise ValueError(f"필수 키 누락: {sorted(missing)}")
    if unknown:
        raise ValueError(f"알 수 없는 키: {sorted(unknown)}")

    name = data["name"]
    active = data["active"]
    score = data["score"]
    tags = data["tags"]

    if not isinstance(name, str):
        raise TypeError("name은 문자열이어야 합니다")
    if not name.strip():
        raise ValueError("name은 비어 있을 수 없습니다")
    if type(active) is not bool:
        raise TypeError("active는 불리언이어야 합니다")
    if type(score) not in (int, float):
        raise TypeError("score는 숫자여야 합니다")
    if isinstance(score, float) and not math.isfinite(score):
        raise ValueError("score는 유한한 수여야 합니다")
    if not 0 <= score <= 100:
        raise ValueError("score는 0부터 100 사이여야 합니다")
    if not isinstance(tags, list):
        raise TypeError("tags는 배열이어야 합니다")
    if any(not isinstance(tag, str) for tag in tags):
        raise TypeError("모든 태그는 문자열이어야 합니다")
    if any(not tag.strip() for tag in tags):
        raise ValueError("태그는 비어 있을 수 없습니다")

    return {
        "name": name.strip(),
        "active": active,
        "score": score,
        "tags": [tag.strip() for tag in tags],
    }


validated_record = validate_record(loads_strict(record_text))
assert validated_record == record

assert "필수 키 누락" in str(expect_exception(
    ValueError,
    validate_record,
    {"name": "Alice", "active": True, "score": 91},
))
assert "숫자" in str(expect_exception(
    TypeError,
    validate_record,
    {"name": "Alice", "active": True, "score": True, "tags": []},
))
assert "0부터 100" in str(expect_exception(
    ValueError,
    validate_record,
    {"name": "Alice", "active": True, "score": 101, "tags": []},
))

print(validated_record)


### 7. 입력 값을 바꾸어 다시 검증

점수와 태그 값을 바꾸되 같은 스키마를 유지합니다. score를 True나 101로 바꾸면 앞 셀의 어느 검사에서 거부되는지 확인해 보세요.


In [ ]:
changed_record = {
    "name": "  민준  ",
    "active": False,
    "score": 88.5,
    "tags": [" json ", "validation"],
}
normalized_changed_record = validate_record(changed_record)

assert normalized_changed_record == {
    "name": "민준",
    "active": False,
    "score": 88.5,
    "tags": ["json", "validation"],
}
assert changed_record["name"] == "  민준  "

print(normalized_changed_record)


### 8. 엄격한 JSON 출력과 크기 경계

allow_nan=False로 비표준 숫자 출력을 막습니다. 전체 JSON 문서를 읽기 전에는 파일 크기 상한도 확인합니다.


In [ ]:
nan_output_error = expect_exception(
    ValueError,
    json.dumps,
    {"score": float("nan")},
    allow_nan=False,
)
assert "compliant" in str(nan_output_error)


def load_small_json(path, *, max_bytes):
    size = path.stat().st_size
    if size > max_bytes:
        raise ValueError(
            f"허용된 JSON 크기를 초과했습니다: {size} > {max_bytes}"
        )
    with path.open("r", encoding="utf-8") as file:
        return load_strict(file)


assert load_small_json(
    record_path,
    max_bytes=record_path.stat().st_size,
) == record
size_error = expect_exception(
    ValueError,
    load_small_json,
    record_path,
    max_bytes=1,
)
assert "크기" in str(size_error)

print("엄격 출력과 파일 크기 경계를 확인했습니다.")


### 9. JSON Lines 행별 검증과 오류 격리

JSON Lines는 한 물리 행에 JSON 값 하나를 저장합니다. 빈 행·긴 행·문법·중복 키·비표준 숫자·스키마 오류를 행별로 분리하고 입력 전문은 오류 결과에 남기지 않습니다.


In [ ]:
ALLOWED_RESULTS = {"FAIL", "SUCCESS"}
MAX_LINE_CHARS = 10_000


def validate_event(data):
    if not isinstance(data, dict):
        raise TypeError("이벤트는 object여야 합니다")
    if set(data) != {"username", "result"}:
        raise ValueError("이벤트 키 구성이 올바르지 않습니다")

    username = data["username"]
    result = data["result"]

    if not isinstance(username, str):
        raise TypeError("username은 문자열이어야 합니다")
    if not username.strip():
        raise ValueError("username은 비어 있을 수 없습니다")
    if not isinstance(result, str):
        raise TypeError("result는 문자열이어야 합니다")
    if result not in ALLOWED_RESULTS:
        raise ValueError("알 수 없는 인증 결과입니다")

    return {
        "username": username.strip(),
        "result": result,
    }


def load_events(path):
    valid_events = []
    errors = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            text = line.rstrip("\r\n")

            if not text:
                errors.append({
                    "line": line_number,
                    "error": "빈 행입니다",
                })
                continue
            if len(text) > MAX_LINE_CHARS:
                errors.append({
                    "line": line_number,
                    "error": "허용된 행 길이를 초과했습니다",
                })
                continue

            try:
                event = validate_event(loads_strict(text))
            except (json.JSONDecodeError, TypeError, ValueError) as exc:
                errors.append({
                    "line": line_number,
                    "error": str(exc),
                })
                continue

            valid_events.append(event)

    return valid_events, errors


In [ ]:
events_path = lab_dir / "events.jsonl"
event_lines = [
    json.dumps(
        {"username": "alice", "result": "FAIL"},
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ),
    "",
    '{"username":"bob","result":}',
    '{"username":"carol"}',
    '{"username":"dave","username":"admin","result":"SUCCESS"}',
    '{"username":"erin","result":NaN}',
    "x" * (MAX_LINE_CHARS + 1),
    json.dumps(
        {"username": "frank", "result": "SUCCESS"},
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ),
]
events_path.write_text(
    "\n".join(event_lines) + "\n",
    encoding="utf-8",
)

valid_events, event_errors = load_events(events_path)

assert valid_events == [
    {"username": "alice", "result": "FAIL"},
    {"username": "frank", "result": "SUCCESS"},
]
assert [item["line"] for item in event_errors] == [2, 3, 4, 5, 6, 7]
assert all(set(item) == {"line", "error"} for item in event_errors)
assert all(
    event_lines[item["line"] - 1] not in item["error"]
    for item in event_errors
    if event_lines[item["line"] - 1]
)

print("정상:", valid_events)
print("오류 위치:", [item["line"] for item in event_errors])


## Checks

### 10. 검증 결과 보고서 저장·복원

summary, records, errors의 구조와 처리 건수 보존식을 검사한 뒤 UTF-8 JSON으로 저장합니다. 같은 정상 레코드는 전용 스키마를 적용해 JSON Lines로도 저장하고 한 줄씩 다시 검증합니다.


In [ ]:
REPORT_KEYS = {"summary", "records", "errors"}
SUMMARY_KEYS = {"total", "valid", "errors", "quarantined"}


def validate_report(report):
    if not isinstance(report, dict):
        raise TypeError("보고서는 object여야 합니다")
    if set(report) != REPORT_KEYS:
        raise ValueError("보고서 키 구성이 올바르지 않습니다")

    summary = report["summary"]
    records = report["records"]
    errors = report["errors"]

    if not isinstance(summary, dict) or set(summary) != SUMMARY_KEYS:
        raise ValueError("summary 키 구성이 올바르지 않습니다")
    if not isinstance(records, list) or not isinstance(errors, list):
        raise TypeError("records와 errors는 배열이어야 합니다")

    for name, count in summary.items():
        if type(count) is not int:
            raise TypeError(f"{name} 건수는 정수여야 합니다")
        if count < 0:
            raise ValueError(f"{name} 건수는 0 이상이어야 합니다")

    if summary["total"] != (
        summary["valid"]
        + summary["errors"]
        + summary["quarantined"]
    ):
        raise ValueError("처리 건수 합계가 일치하지 않습니다")
    if summary["valid"] != len(records):
        raise ValueError("정상 건수와 records 길이가 다릅니다")
    if summary["errors"] + summary["quarantined"] != len(errors):
        raise ValueError("오류 건수와 errors 길이가 다릅니다")

    return report


report = {
    "summary": {
        "total": 8,
        "valid": 2,
        "errors": 5,
        "quarantined": 1,
    },
    "records": [
        {"name": "Notebook", "price": 1000, "quantity": 2, "total": 2000},
        {"name": "USB, Cable", "price": 500, "quantity": 3, "total": 1500},
    ],
    "errors": [
        {"record": 3, "error": "빈 필드", "kind": "error"},
        {"record": 4, "error": "정수 형식", "kind": "error"},
        {"record": 5, "error": "범위 오류", "kind": "error"},
        {"record": 6, "error": "필드 누락", "kind": "error"},
        {"record": 7, "error": "초과 필드", "kind": "error"},
        {"record": 8, "error": "수식 위험", "kind": "quarantined"},
    ],
}

assert validate_report(report) is report
broken_report = {
    **report,
    "summary": {**report["summary"], "total": 9},
}
report_error = expect_exception(
    ValueError, validate_report, broken_report
)
assert "합계" in str(report_error)


In [ ]:
report_path = lab_dir / "validation-report.json"

with report_path.open("w", encoding="utf-8") as file:
    json.dump(
        validate_report(report),
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )
    file.write("\n")

with report_path.open("r", encoding="utf-8") as file:
    restored_report = validate_report(load_strict(file))

assert restored_report == report
assert report_path.read_bytes().endswith(b"\n")
print(restored_report["summary"])


In [ ]:
PRODUCT_KEYS = {"name", "price", "quantity", "total"}


def validate_product_json_record(data):
    if not isinstance(data, dict) or set(data) != PRODUCT_KEYS:
        raise ValueError("상품 레코드 키 구성이 올바르지 않습니다")
    if not isinstance(data["name"], str) or not data["name"].strip():
        raise ValueError("상품명은 비어 있지 않은 문자열이어야 합니다")
    for key in ("price", "quantity", "total"):
        if type(data[key]) is not int or data[key] < 0:
            raise ValueError(f"{key}는 0 이상의 정수여야 합니다")
    if data["total"] != data["price"] * data["quantity"]:
        raise ValueError("상품 합계가 일치하지 않습니다")
    return data


records_path = lab_dir / "valid-products.jsonl"
with records_path.open("w", encoding="utf-8") as file:
    for product in report["records"]:
        validate_product_json_record(product)
        line = json.dumps(
            product,
            ensure_ascii=False,
            allow_nan=False,
            separators=(",", ":"),
        )
        file.write(line + "\n")

restored_products = []
with records_path.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        assert line.endswith("\n")
        restored_products.append(
            validate_product_json_record(loads_strict(line))
        )

assert restored_products == report["records"]
assert len(restored_products) == report["summary"]["valid"]

print("JSON Lines 정상 레코드:", len(restored_products))


In [ ]:
final_checks = {
    "tuple의 왕복 손실 확인": isinstance(restored_source["position"], list),
    "문법 오류 위치 보존": syntax_error_summary["line"] >= 1,
    "중복 키 거부": "중복 JSON 키" in str(duplicate_error),
    "비표준 숫자 입력 거부": "NaN" in str(nan_input_error),
    "비표준 숫자 출력 거부": isinstance(nan_output_error, ValueError),
    "스키마에서 bool과 숫자 구분": type(validated_record["score"]) is int,
    "JSONL 행별 오류 격리": len(event_errors) == 6,
    "보고서 건수 보존": (
        restored_report["summary"]["total"]
        == restored_report["summary"]["valid"]
        + restored_report["summary"]["errors"]
        + restored_report["summary"]["quarantined"]
    ),
    "JSONL 출력 재검증": restored_products == report["records"],
}

for name, passed in final_checks.items():
    assert passed
    print(f"[PASS] {name}")


## Next Steps

- JSON 문법 검증과 업무 스키마 검증을 별도 단계로 설명합니다.
- 중복 키·NaN·Infinity·직렬화 불가 자료형에 명시적인 정책을 적용합니다.
- JSON Lines 오류에는 위치와 일반화한 원인만 남기고 전체 입력을 노출하지 않습니다.
- 큰 단일 JSON은 읽기 전에 크기 상한을 확인하고, 큰 레코드 모음은 다음 절의 스트리밍 처리로 확장합니다.
- 신뢰하지 않는 pickle은 절대 역직렬화하지 않습니다.


In [ ]:
json_tempdir.cleanup()
assert not lab_dir.exists()
print("임시 실습 디렉터리를 정리했습니다.")
